In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, round, lag, avg, to_date,
    unix_timestamp, lit, when
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, DateType

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
from pyspark.sql.functions import lit

stocks = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]

df_list = []
for stock in stocks:
    path = f"/mnt/bronze/stock_data/{stock}_daily_stock.csv"
    df_temp = spark.read.option("header", True).csv(path)
    # Add symbol column HERE before union
    df_temp = df_temp.withColumn("symbol", lit(stock))
    df_list.append(df_temp)

# Combine all into one dataframe
df_raw = df_list[0]
for df in df_list[1:]:
    df_raw = df_raw.union(df)

print(f"Total rows loaded: {df_raw.count():,}")
print(f"Columns: {df_raw.columns}")
df_raw.show(5)

Total rows loaded: 500
Columns: ['timestamp', 'open', 'high', 'low', 'close', 'volume', 'symbol']
+----------+--------+--------+--------+--------+--------+------+
| timestamp|    open|    high|     low|   close|  volume|symbol|
+----------+--------+--------+--------+--------+--------+------+
|2026-03-19|249.4000|251.8300|247.3000|248.9600|34864082|  AAPL|
|2026-03-18|252.6250|254.9400|249.0000|249.9400|35757874|  AAPL|
|2026-03-17|252.9550|255.1300|252.1800|254.2300|32361607|  AAPL|
|2026-03-16|252.1050|253.8850|249.8800|252.8200|32074209|  AAPL|
|2026-03-13|255.4800|256.3300|249.5200|250.1200|36929988|  AAPL|
+----------+--------+--------+--------+--------+--------+------+
only showing top 5 rows



In [0]:
# Fix data types and clean column names
df_clean = df_raw.select(
    to_date(col("timestamp")).alias("date"),
    col("symbol"),
    col("open").cast(DoubleType()).alias("open"),
    col("high").cast(DoubleType()).alias("high"),
    col("low").cast(DoubleType()).alias("low"),
    col("close").cast(DoubleType()).alias("close"),
    col("volume").cast(DoubleType()).alias("volume")
)

# Drop any rows with nulls
df_clean = df_clean.dropna()

print(f"Rows after cleaning: {df_clean.count():,}")
print(f"Null values remaining: {df_clean.filter(col('close').isNull()).count()}")
df_clean.show(5)

Rows after cleaning: 500
Null values remaining: 0
+----------+------+-------+-------+------+------+-----------+
|      date|symbol|   open|   high|   low| close|     volume|
+----------+------+-------+-------+------+------+-----------+
|2026-03-19|  AAPL|  249.4| 251.83| 247.3|248.96|3.4864082E7|
|2026-03-18|  AAPL|252.625| 254.94| 249.0|249.94|3.5757874E7|
|2026-03-17|  AAPL|252.955| 255.13|252.18|254.23|3.2361607E7|
|2026-03-16|  AAPL|252.105|253.885|249.88|252.82|3.2074209E7|
|2026-03-13|  AAPL| 255.48| 256.33|249.52|250.12|3.6929988E7|
+----------+------+-------+-------+------+------+-----------+
only showing top 5 rows



In [0]:
# cell 4
# Define window per stock ordered by date
window_spec = Window.partitionBy("symbol").orderBy("date")

# Add calculated columns
df_enriched = df_clean \
    .withColumn(
        "prev_close",
        lag("close", 1).over(window_spec)
    ) \
    .withColumn(
        "daily_return_pct",
        round(
            ((col("close") - col("prev_close")) / col("prev_close")) * 100,
            2
        )
    ) \
    .withColumn(
        "price_change",
        round(col("close") - col("prev_close"), 2)
    ) \
    .withColumn(
        "moving_avg_7day",
        round(avg("close").over(
            window_spec.rowsBetween(-6, 0)
        ), 2)
    ) \
    .withColumn(
        "daily_range",
        round(col("high") - col("low"), 2)
    ) \
    .withColumn(
        "is_positive_day",
        when(col("daily_return_pct") > 0, "Yes").otherwise("No")
    ) \
    .drop("prev_close")

print("Derived columns added!")
print(f"Total columns now: {len(df_enriched.columns)}")
df_enriched.show(5)

Derived columns added!
Total columns now: 12
+----------+------+-------+------+--------+------+-----------+----------------+------------+---------------+-----------+---------------+
|      date|symbol|   open|  high|     low| close|     volume|daily_return_pct|price_change|moving_avg_7day|daily_range|is_positive_day|
+----------+------+-------+------+--------+------+-----------+----------------+------------+---------------+-----------+---------------+
|2025-10-24|  AAPL| 261.19|264.13|  259.18|262.82|3.8253717E7|            NULL|        NULL|         262.82|       4.95|             No|
|2025-10-27|  AAPL| 264.88|269.12|264.6501|268.81|4.4888152E7|            2.28|        5.99|         265.82|       4.47|            Yes|
|2025-10-28|  AAPL|268.985|269.89|  268.15| 269.0|4.1534759E7|            0.07|        0.19|         266.88|       1.74|            Yes|
|2025-10-29|  AAPL|269.275|271.41|  267.11| 269.7|5.1086742E7|            0.26|         0.7|         267.58|        4.3|            Y

In [0]:
# cell 5
print("=== Silver Layer Summary ===")
print(f"Total rows: {df_enriched.count():,}")
print(f"Stocks covered: {df_enriched.select('symbol').distinct().count()}")
print(f"Date range: ", end="")

date_range = df_enriched.agg(
    {"date": "min"}
).collect()[0][0]
print(date_range)

print("\nSample per stock:")
df_enriched.groupBy("symbol").count().orderBy("symbol").show()

=== Silver Layer Summary ===
Total rows: 500
Stocks covered: 5
Date range: 2025-10-24

Sample per stock:
+------+-----+
|symbol|count|
+------+-----+
|  AAPL|  100|
|  AMZN|  100|
| GOOGL|  100|
|  MSFT|  100|
|  TSLA|  100|
+------+-----+



In [0]:
# cell 6
# Write clean enriched data to Silver layer
df_enriched.write \
    .mode("overwrite") \
    .partitionBy("symbol") \
    .parquet("/mnt/silver/stock_data/")

print("Silver layer written successfully!")

# Verify files were created
files = dbutils.fs.ls("/mnt/silver/stock_data/")
print("\nFolders in Silver container:")
for f in files:
    print(f"  {f.name}")

Silver layer written successfully!

Folders in Silver container:
  _SUCCESS
  symbol=AAPL/
  symbol=AMZN/
  symbol=GOOGL/
  symbol=MSFT/
  symbol=TSLA/


In [0]:
# cell 7
# Read back from Silver to confirm it saved correctly
df_silver = spark.read.parquet("/mnt/silver/stock_data/")

print("=== Silver Layer Verification ===")
print(f"Total rows: {df_silver.count():,}")
print(f"Columns: {df_silver.columns}")
print(f"\nSample data:")
df_silver.orderBy("symbol", "date").show(10)

=== Silver Layer Verification ===
Total rows: 500
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'daily_return_pct', 'price_change', 'moving_avg_7day', 'daily_range', 'is_positive_day', 'symbol']

Sample data:
+----------+-------+-------+--------+------+-----------+----------------+------------+---------------+-----------+---------------+------+
|      date|   open|   high|     low| close|     volume|daily_return_pct|price_change|moving_avg_7day|daily_range|is_positive_day|symbol|
+----------+-------+-------+--------+------+-----------+----------------+------------+---------------+-----------+---------------+------+
|2025-10-24| 261.19| 264.13|  259.18|262.82|3.8253717E7|            NULL|        NULL|         262.82|       4.95|             No|  AAPL|
|2025-10-27| 264.88| 269.12|264.6501|268.81|4.4888152E7|            2.28|        5.99|         265.82|       4.47|            Yes|  AAPL|
|2025-10-28|268.985| 269.89|  268.15| 269.0|4.1534759E7|            0.07|        0.19| 